## 3.0 AGN Time Lags metric

The AGN TimeLag metric looks at the timing of visits and calculates the (mean, min or max) of the interval between visits. This difference is then compared to the time interval required for nyquist sampling over the 'time lag' period. The value of time lag / (1 + z + time interval is returned, if it is above the nyquist threshold of 2.2.

There are modest magnitude limit cuts in g and r bands before a visit is counted, but no other bandpasses. Most short visits will be counted.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
import rubin_sim.maf as maf
from rubin_sim.data import get_baseline

## 1.0 Configuration

In [ ]:
baseline_file = get_baseline()
run_name = os.path.split(baseline_file)[-1].replace(".db", "")
print(f"{run_name} : {baseline_file}")

In [ ]:
data_dir = None

if data_dir is None:
    import tempfile
    import os

    # Scratch directory for MAF's results database and any output products from this notebook.
    data_dir_itself = tempfile.TemporaryDirectory(prefix="02_maf_AGN_StructFunc_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

# Set up output directories
out_dir = data_dir
resultsDb = maf.ResultsDb(out_dir)

## 2. Creation of slicer and Metrucs and Bundle

In [ ]:
# Run the TimeLag for each filter *and* all filters
nside = 64
filterlist, colors, filterorders, filtersqls, filterinfo_label = maf.filter_list(
    all=True,
)
slicer = maf.HealpixSlicer(nside=nside, use_cache=False)
dustmap = maf.DustMap(nside=nside)
display_dict = {}
bundleList = []

nquist_threshold = 2.2
lag = 100
summaryMetrics = maf.extended_summary()
summaryMetrics += [maf.AreaThresholdMetric(lower_threshold=nquist_threshold)]
m = maf.AgnTimeLagMetric(threshold=nquist_threshold, lag=lag)
for f in filterlist:
    plot_dict = {"color": colors[f], "color_min": 0, "color_max": 5, "percentile_clip": 95}
    display_dict["order"] = filterorders[f]
    display_dict["subgroup"] = "Time Lags"
    display_dict["caption"] = (
        f"Comparion of the time between visits compared to a defined sampling gap ({lag} days) in "
        f"{f} band."
    )
    bundleList.append(
        maf.MetricBundle(
            m,
            slicer,
            constraint=filtersqls[f],
            info_label=filterinfo_label[f],
            run_name=run_name,
            maps_list=[dustmap],
            plot_dict=plot_dict,
            summary_metrics=summaryMetrics,
            display_dict=display_dict,
        )
    )
bundles = maf.make_bundles_dict_from_list(bundleList)

## 3.0 Create group bundle and Run

In [ ]:
g = maf.MetricBundleGroup(bundles, baseline_file, data_dir, None)
g.run_all()

## 4.0 Plot

In [ ]:
for k in bundles:
    bundles[k].plot()

Compare results across a variety of runs.

In [ ]:
# Variance across (most of) current set of simulations?
summaries = maf.get_metric_summaries()
families = maf.get_family_descriptions()
metric_sets = maf.get_metric_sets()

In [ ]:
fams = [f for f in families.index if (not f.startswith("ddf")) & (not "v2.99" in f)]
runs = families.explode(["run"]).loc[fams]["run"]

runs = [run for run in runs if ("draft" not in run.lower()) & ("2.99" not in run)]

In [ ]:
# Plot the *normalized* values
k = "AGN timelag"
fig, ax = maf.plot_run_metric_mesh(
    summaries.loc[runs, metric_sets.loc[k]["metric"]],
    baseline_run="baseline_v2.0_10yrs",
    color_range=0.5,
    metric_label_map=metric_sets.loc[k]["short_name"],
    metric_set=metric_sets.loc[k],
)
fig.set_figwidth(18)

In [ ]:
# Plot the normalized values, in a different way
fig, ax = maf.plot_run_metric(
    summaries.loc[runs, metric_sets.loc[k]["metric"]],
    baseline_run="baseline_v2.0_10yrs",
    metric_label_map=metric_sets.loc[k]["short_name"],
    metric_set=metric_sets.loc[k],
    horizontal_quantity="value",
    vertical_quantity="run",
)
fig.set_figheight(25)
ax.legend(loc=(1.01, 0.5))
ax.set_xlim(0.5, 1.5)